In [1]:
import numpy as np
import pandas as pd
import glob
import re
import os
from plotnine import *

In [6]:
# import reward simulations
csv_files =  (
    glob.glob('../data/rl_policies/size-time/*.csv') + glob.glob('../data/rl_policies/count/*.csv') +
    glob.glob('../data/rl_policies/count-biomass-time/*.csv') + glob.glob('../data/rl_policies/count-time/*.csv')
)

dfs = []
for filepath in csv_files:
    filename = os.path.basename(filepath)  # e.g., 'td3_count-time_sim_3.csv'
    
    # Parse filename: {algorithm}_{obs_type}_sim_{replicate}.csv
    match = re.match(r'([^_]+)_(.+)_sim_(\d+)\.csv', filename)
    if match:
        algorithm = match.group(1)   # e.g., 'td3'
        obs_type  = match.group(2)   # e.g., 'count-time'
        replicate = int(match.group(3))  # e.g., 3
    else:
        print(f"Skipping unrecognized filename: {filename}")
        continue
    
    df = pd.read_csv(filepath)
    df = df[df['t'] == 99]  # subset by t value
    
    df['algorithm'] = algorithm
    df['obs_type']  = obs_type
    df['replicate'] = replicate
    
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

In [7]:
combined_df.head()

,t,crabs,act0,act1,rew,rep,crab_pop,nonlocal_crab,algorithm,obs_type,replicate,months
0,99,[-1. -1. -1. -1. ...,-1.0,1.0,-12.290053,0,[2.66324005e-23 1.19111981e-16 3.35881924e-11 ...,[84542.24134574 7477.7500429 77125.72627453 ...,td3,size-time,4,NaN
1,99,[-1. -1. -1. -1. ...,-1.0,1.0,-13.807641,1,[1.37139565e-38 3.92449459e-28 2.46035852e-19 ...,[84542.24134574 7477.7500429 77125.72627453 ...,td3,size-time,4,NaN
2,99,[-1. -1. -1. -1. ...,-1.0,1.0,-10.064081,2,[1.41405318e-24 1.45946613e-17 9.37504328e-12 ...,[84542.24134574 7477.7500429 77125.72627453 ...,td3,size-time,4,NaN
3,99,[-1. -1. -1. -1. ...,-1.0,1.0,-9.003513,3,[1.74292396e-21 2.40197950e-15 2.53611443e-10 ...,[84542.24134574 7477.7500429 77125.72627453 ...,td3,size-time,4,NaN
4,99,[-1. -1. -1. -1. ...,-1.0,1.0,-10.549199,4,[1.80833582e-17 9.43833965e-12 2.80716335e-07 ...,[84542.24134574 7477.7500429 77125.72627453 ...,td3,size-time,4,NaN


In [8]:
# 1. Group by algorithm and obs_type, mean and variance of rew
summary1 = (combined_df
    .groupby(['algorithm', 'obs_type'])['rew']
    .agg(mean_rew='mean', var_rew='var')
    .reset_index()
)

# 2. First average within each replicate, then mean and variance across replicates
summary2 = (combined_df
    .groupby(['algorithm', 'obs_type', 'replicate'])['rew']
    .mean()
    .reset_index()
    .rename(columns={'rew': 'mean_rew'})
    .groupby(['algorithm', 'obs_type'])['mean_rew']
    .agg(mean_rew='mean', var_rew='var')
    .reset_index()
)

In [9]:
summary1

,algorithm,obs_type,mean_rew,var_rew
0,ppo,count,-7.839712,3.226293
1,ppo,count-biomass-time,-7.655120,6.419178
2,ppo,count-time,-8.556171,9.003671
3,ppo,size-time,-8.194557,3.316351
4,td3,count,-12.406705,39.291344
5,td3,count-biomass-time,-7.920903,3.431521
6,td3,count-time,-16.177410,38.445420
7,td3,size-time,-13.221938,26.929622
8,tqc,count,-9.512286,10.418472
9,tqc,count-biomass-time,-7.533753,2.907288


In [10]:
summary2

,algorithm,obs_type,mean_rew,var_rew
0,ppo,count,-7.839712,0.247467
1,ppo,count-biomass-time,-7.655120,1.379700
2,ppo,count-time,-8.556171,2.480843
3,ppo,size-time,-8.194557,0.372903
4,td3,count,-12.406705,36.575769
5,td3,count-biomass-time,-7.920903,0.352993
6,td3,count-time,-16.177410,32.513934
7,td3,size-time,-13.221938,28.103821
8,tqc,count,-9.512286,8.001038
9,tqc,count-biomass-time,-7.533753,0.028638


In [13]:
# get best TQC replicate
(
    combined_df[combined_df['algorithm'] == 'tqc'].
    groupby(['algorithm', 'obs_type', 'replicate'])['rew'].mean().idxmax()
)


('tqc', 'count-biomass-time', np.int64(3))